## Shower Validation

written by Isobel Mawby (i.mawby1@lancaster.ac.uk)

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Imports
</div>

In [ ]:
import random
import uproot
import numpy as np
import math
import matplotlib.pyplot as plt
import awkward as ak

%matplotlib widget
from termcolor import colored, cprint

import Definitions
import ValidationFunc

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Config
</div>

In [ ]:
SHOW_PLOTS = True

<div class="alert alert-block alert-info" style="font-size: 18px;">
    File
</div>

In [ ]:
file_name = "/Users/isobel/Desktop/DUNE/2026/PandoraValidation/files/ValidationVis_nu.root"
plot_dir = '/Users/isobel/Desktop/DUNE/2026/PandoraValidation/ShowerValPlots/'

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Lets open the file...
</div>

In [ ]:
file = uproot.open(file_name)
shower_tree = file['ShowerTree']
print(shower_tree.keys())


In [ ]:
event_tree = file['EventTree']
pfp_tree = file['PFPTree']
shower_tree = file['ShowerTree']
hierarchy_tree = file['HierarchyTree']

event_branches = event_tree.arrays(['Run', 'Subrun', 'Event', 'MCInt_IsCC', 'MCNu_PDG'], library="ak")
pfp_branches = pfp_tree.arrays(['Run', 'Subrun', 'Event',
                                'MCP_TruePDG', 'MCP_HasMatch',
                                'MCP_NMCHits2D', 'MCP_NMCHitsU', 'MCP_NMCHitsV', 'MCP_NMCHitsW',
                                'BM_Completeness', 'BM_Purity', 'BM_IsTrack', 'BM_IsShower'], library="ak")

shower_branches = shower_tree.arrays(['MCP_TrueCoreLengthFromU', 'MCP_TrueCoreLengthFromV', 'MCP_TrueCoreLengthFromW',
                                      'BM_RecoCoreLength', 'BM_RecoLength', 'BM_MoliereRadius',
                                      'BM_DirAcc',
                                      'MCP_InitialMCHits', 'MCP_InitialMCHitsU', 'MCP_InitialMCHitsV', 'MCP_InitialMCHitsW',
                                      'BM_InitialPfoHits', 'BM_InitialPfoHitsU', 'BM_InitialPfoHitsV', 'BM_InitialPfoHitsW',
                                      'BM_InitialCompleteness', 'BM_InitialCompletenessU', 'BM_InitialCompletenessV', 'BM_InitialCompletenessW',
                                      'BM_InitialPurity', 'BM_InitialPurityU', 'BM_InitialPurityV', 'BM_InitialPurityW'], library="ak")

hierarchy_branches = hierarchy_tree.arrays(['MC_HierarchyTier'], library="ak")

In [ ]:
# entry = 2607

# print(f"Run: {event_branches['Run'][entry]}, Subrun: {event_branches['Subrun'][entry]}, Event: {event_branches['Event'][entry]}")
# print(f"Run: {pfp_branches['Run'][entry]}, Subrun: {pfp_branches['Subrun'][entry]}, Event: {pfp_branches['Event'][entry]}")

for iEntry in range(len(shower_branches['MCP_TrueCoreLengthFromU'])) :
    if (len(shower_branches['MCP_TrueCoreLengthFromU'][iEntry]) != len(pfp_branches['MCP_TruePDG'][iEntry])) :
        print(len(shower_branches['MCP_TrueCoreLengthFromU'][iEntry]))
        print(len(pfp_branches['MCP_TruePDG'][iEntry]))
        print(iEntry)
        break


<div class="alert alert-block alert-info" style="font-size: 18px;">
    Summary
</div>

In [ ]:
int_masks = Definitions.GetIntMasks(event_branches, pfp_branches)
pdg_masks = Definitions.GetPDGMasks(pfp_branches)
tier_masks = Definitions.GetTierMasks(hierarchy_branches)

# Apply custom def of reconstructable and reconstructed
# The tree constains some particles that we reconstructed, even if not initially deemed to be a target
pfp_target_mask = Definitions.GetIsTargetMask(pfp_branches)
pfp_reco_mask = Definitions.GetIsRecoMask(pfp_target_mask, pfp_branches)

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Get plots + tables
</div>

In [ ]:
# MCP branch variables to plot
MCP_plotting_vars = [ValidationFunc.shower_initial_MC_hits, ValidationFunc.shower_initial_PFP_hits, ValidationFunc.shower_initial_completeness, ValidationFunc.shower_initial_purity]

# BM branch variables to plot
BM_plotting_vars = [ValidationFunc.shower_dir_acc, ValidationFunc.shower_moliere]

# Diff variables
diff_plotting_vars = [ValidationFunc.core_length_diff]

for tier in Definitions.tiers :
    tier_mask = tier_masks[tier]

    # MCP branch variable plots
    MCP_var_plots = [plt.subplots(ncols=len(Definitions.pdgs), nrows=len(Definitions.ints), figsize=(14, 10)) for _ in MCP_plotting_vars]
    # BM branch variable plots
    BM_var_plots = [plt.subplots(ncols=len(Definitions.pdgs), nrows=len(Definitions.ints), figsize=(14, 10)) for _ in BM_plotting_vars]
    # Diff branch variable plots
    diff_var_plots = [plt.subplots(ncols=len(Definitions.pdgs), nrows=len(Definitions.ints), figsize=(14, 10)) for _ in diff_plotting_vars]
    
    for int_type in Definitions.ints :
        int_mask = int_masks[int_type]

        for i_pdg in range(len(Definitions.pdgs)) :
            pdg = Definitions.pdgs[i_pdg]
            pdg_mask = pdg_masks[pdg]

            target_mask = pfp_target_mask & tier_mask & int_mask & pdg_mask
            reco_mask = target_mask & pfp_reco_mask

            # Plot MCP_var distributions
            for i_var in range(len(MCP_plotting_vars)) :
                fig, axes = MCP_var_plots[i_var]
                ax = axes[int_type, i_pdg]
                ValidationFunc.ConfigurePlot(fig, ax, int_type, tier, MCP_plotting_vars[i_var])                
                ValidationFunc.PlotVariable(target_mask, shower_branches, MCP_plotting_vars[i_var], ax, Definitions.pdg_strings[pdg], Definitions.pdg_color[pdg])
                if not SHOW_PLOTS :
                    plt.close(fig)

            # Plot BM_var distributions
            for i_var in range(len(BM_plotting_vars)) :
                fig, axes = BM_var_plots[i_var]
                ax = axes[int_type, i_pdg]
                ValidationFunc.ConfigurePlot(fig, ax, int_type, tier, BM_plotting_vars[i_var])
                ValidationFunc.PlotVariable(reco_mask, shower_branches, BM_plotting_vars[i_var], ax, Definitions.pdg_strings[pdg], Definitions.pdg_color[pdg])
                if not SHOW_PLOTS :
                    plt.close(fig)      

            # Plot diff_vars
            for i_var in range(len(diff_plotting_vars)) :
                fig, axes = diff_var_plots[i_var]
                ax = axes[int_type, i_pdg]
                ValidationFunc.ConfigurePlot(fig, ax, int_type, tier, diff_plotting_vars[i_var])
                ValidationFunc.PlotDiffVariable(reco_mask, shower_branches, diff_plotting_vars[i_var], ax, Definitions.pdg_strings[pdg], Definitions.pdg_color[pdg])
                if not SHOW_PLOTS :
                    plt.close(fig)                   

    # Save MCP_var distributions
    for i_var in range(len(MCP_plotting_vars)) :
        fig, _ = MCP_var_plots[i_var]
        file_name = f'{MCP_plotting_vars[i_var].tree_name}_{Definitions.tier_strings[tier]}'
        fig.savefig(f'{plot_dir}{file_name}.pdf', bbox_inches='tight')

    # Save BM_var distributions
    for i_var in range(len(BM_plotting_vars)) :
        fig, _ = BM_var_plots[i_var]
        file_name = f'{BM_plotting_vars[i_var].tree_name}_{Definitions.tier_strings[tier]}'
        fig.savefig(f'{plot_dir}{file_name}.pdf', bbox_inches='tight')        

    # Save diff var distributions
    for i_var in range(len(diff_plotting_vars)) :
        fig, _ = diff_var_plots[i_var]
        file_name = f'{diff_plotting_vars[i_var].true_tree_name}-{diff_plotting_vars[i_var].reco_tree_name}'
        fig.savefig(f'{plot_dir}{file_name}.pdf', bbox_inches='tight') 
